# 1. Exploração Inicial e Diagnóstico de Qualidade

Antes de qualquer transformação, é essencial entender o estado real dos dados:
- **Nulos**: colunas com valores ausentes que podem afetar os cálculos
- **Duplicatas**: registros repetidos que distorcem métricas
- **Tipos de dados**: colunas de data armazenadas como texto precisam de conversão

In [5]:
import pandas as pd

In [6]:

df_olist_customers = pd.read_csv("../data/raw/olist_customers_dataset.csv")
df_olist_geolocation = pd.read_csv("../data/raw/olist_geolocation_dataset.csv")
olist_order_items = pd.read_csv("../data/raw/olist_order_items_dataset.csv")
olist_order_payments = pd.read_csv("../data/raw/olist_order_payments_dataset.csv")
olist_order_reviews = pd.read_csv("../data/raw/olist_order_reviews_dataset.csv")
olist_orders = pd.read_csv("../data/raw/olist_orders_dataset.csv")
olist_products = pd.read_csv("../data/raw/olist_products_dataset.csv")
olist_sellers = pd.read_csv("../data/raw/olist_sellers_dataset.csv")
product_category_name_translation = pd.read_csv("../data/raw/product_category_name_translation.csv")

In [7]:

# Reunindo todas as bases em um dicionário para facilitar a iteração
# Isso evita repetir o mesmo código para cada dataframe — boa prática!
dataframes = {
    "Clientes":               df_olist_customers,
    "Geolocalização":         df_olist_geolocation,
    "Itens dos Pedidos":      olist_order_items,
    "Pagamentos":             olist_order_payments,
    "Avaliações":             olist_order_reviews,
    "Pedidos":                olist_orders,
    "Produtos":               olist_products,
    "Vendedores":             olist_sellers,
    "Tradução de Categorias": product_category_name_translation,
}

In [8]:
# Função de diagnóstico reutilizável
def diagnostico(nome_tabela, df):
    print(f"{'='*55}")
    print(f" {nome_tabela}")
    print(f" Linhas       : {df.shape[0]:,}")
    print(f" Colunas      : {df.shape[1]}")
    
    # Nulos
    nulos = df.isnull().sum() # conta os nulos em cada coluna
    nulos = nulos[nulos > 0] # filtra só as colunas QUE TÊM nulos
    if nulos.empty: # se depois do filtro ficou vazio = não tem nulos
        print(" Nulos        : ✅ Nenhum valor nulo encontrado")  # → tudo ok!
    else:
        print(" Nulos        : ⚠️  Colunas com valores nulos:") # → encontrou colunas com nulos
        for col, qtd in nulos.items():
            pct = qtd / len(df) * 100
            print(f"   → {col}: {qtd:,} ({pct:.1f}%)")
    
    # Duplicatas
    dup = df.duplicated().sum() # conta quantas linhas são duplicatas
    if dup == 0:  # se zero → nenhuma duplicata
        print(" Duplicatas   : ✅ Nenhuma linha duplicada")
    else: # se maior que zero → tem duplicatas
        print(f" Duplicatas   : ⚠️  {dup:,} linhas duplicadas ({dup/len(df)*100:.1f}%)")
    print()

In [9]:
for nome, df in dataframes.items():
    diagnostico(nome, df)

 Clientes
 Linhas       : 99,441
 Colunas      : 5
 Nulos        : ✅ Nenhum valor nulo encontrado
 Duplicatas   : ✅ Nenhuma linha duplicada

 Geolocalização
 Linhas       : 1,000,163
 Colunas      : 5
 Nulos        : ✅ Nenhum valor nulo encontrado
 Duplicatas   : ⚠️  261,831 linhas duplicadas (26.2%)

 Itens dos Pedidos
 Linhas       : 112,650
 Colunas      : 7
 Nulos        : ✅ Nenhum valor nulo encontrado
 Duplicatas   : ✅ Nenhuma linha duplicada

 Pagamentos
 Linhas       : 103,886
 Colunas      : 5
 Nulos        : ✅ Nenhum valor nulo encontrado
 Duplicatas   : ✅ Nenhuma linha duplicada

 Avaliações
 Linhas       : 99,224
 Colunas      : 7
 Nulos        : ⚠️  Colunas com valores nulos:
   → review_comment_title: 87,656 (88.3%)
   → review_comment_message: 58,247 (58.7%)
 Duplicatas   : ✅ Nenhuma linha duplicada

 Pedidos
 Linhas       : 99,441
 Colunas      : 8
 Nulos        : ⚠️  Colunas com valores nulos:
   → order_approved_at: 160 (0.2%)
   → order_delivered_carrier_date: 1,783 

In [10]:
# Investigando as duplicatas da tabela de Geolocalização
# Aqui queremos ver quantas vezes cada LINHA INTEIRA se repete

# keep=False → marca TODAS as ocorrências da linha duplicada (não só a segunda)
linhas_duplicadas = df_olist_geolocation[df_olist_geolocation.duplicated(keep=False)]

# Agrupa todas as colunas e conta quantas vezes cada combinação aparece
contagem_linhas = (
    linhas_duplicadas
    .groupby(list(df_olist_geolocation.columns))  # agrupa por todas as colunas
    .size()                                        # conta as ocorrências
    .reset_index(name="quantidade")               # transforma em coluna
    .sort_values("quantidade", ascending=False)   # ordena do mais repetido
)

print(f"Total de linhas duplicadas: {len(linhas_duplicadas):,}")
print(f"Combinações únicas que se repetem: {len(contagem_linhas):,}\n")
contagem_linhas

Total de linhas duplicadas: 390,005
Combinações únicas que se repetem: 128,174



,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state,quantidade
116830,88220,-27.102099,-48.629613,itapema,SC,314
23250,6414,-23.495901,-46.874687,barueri,SP,189
23259,6414,-23.490618,-46.869004,barueri,SP,127
17827,5145,-23.506049,-46.717377,sao paulo,SP,126
59852,22620,-23.005514,-43.375964,rio de janeiro,RJ,102
...,...,...,...,...,...,...
53657,20720,-22.911518,-43.288969,rio de janeiro,RJ,2
53655,20715,-22.908708,-43.270064,rio de janeiro,RJ,2
53653,20715,-22.912357,-43.270797,rio de janeiro,RJ,2
53652,20715,-22.912505,-43.271769,rio de janeiro,RJ,2


In [11]:
# ============================================================
# VISUALIZAÇÃO DAS PRIMEIRAS LINHAS DAS TABELAS PRINCIPAIS
# ============================================================
# A tabela de orders é a tabela central — todas as análises
# partem dela. Verificamos as colunas e tipos de dados.
# ============================================================

print('=== TABELA: orders (tabela central) ===')
print(olist_orders.dtypes)
print()
olist_orders.head(3)

=== TABELA: orders (tabela central) ===
order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object



,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00


In [ ]:
# ============================================================
# DISTRIBUIÇÃO DE STATUS DOS PEDIDOS
# ============================================================
# Entender o mix de status é fundamental para decidir quais pedidos incluir nas análises de entrega.
# Só pedidos 'delivered' têm data de entrega real.
# ============================================================
# Significado de cada status:
#   delivered   → pedido entregue ao cliente (único com data de entrega real)
#   shipped     → pedido enviado à transportadora, aguardando entrega
#   canceled    → pedido cancelado antes ou depois do envio
#   unavailable → produto indisponível, pedido não pode ser processado
#   invoiced    → nota fiscal emitida, aguardando envio
#   processing  → pagamento aprovado, pedido em preparação pelo vendedor
#   created     → pedido criado mas pagamento ainda não confirmado
#   approved    → pagamento aprovado manualmente, ainda não entrou em processamento
# ============================================================

status_dist = olist_orders['order_status'].value_counts()
print('Distribuição de status dos pedidos:')
for status, count in status_dist.items():
    pct = count / len(olist_orders) * 100
    print(f'  {status:<30} {count:>6,}  ({pct:.1f}%)')

Distribuição de status dos pedidos:
  delivered                      96,478  (97.0%)
  shipped                         1,107  (1.1%)
  canceled                          625  (0.6%)
  unavailable                       609  (0.6%)
  invoiced                          314  (0.3%)
  processing                        301  (0.3%)
  created                             5  (0.0%)
  approved                            2  (0.0%)
